# Public review copy — MIS581 Capstone

This notebook is a sanitized publication copy of an executed development notebook. Local Windows paths were replaced with relative placeholders and embedded Census API configuration was changed to use the `CENSUS_API_KEY` environment variable. Historical error outputs were omitted; successful saved outputs were retained. The notebook has **not** been independently re-executed as a clean installation. It may require original source data, a compatible Python environment, and (for Notebook 08) PostgreSQL. Use the project README to understand dependencies and expected folder layout.


In [1]:
import pandas as pd

file_path = r"../data/raw/cbsaannual_202599.xls"

df_permits = pd.read_excel(
    file_path,
    header=5
)

print(df_permits.head())
print(df_permits.shape)
print(df_permits.dtypes)
df_permits.info()

     CSA     CBSA                                               Name  \
0    NaN      NaN                                                NaN   
1  999.0  10100.0  Aberdeen, SD                                  ...   
2  999.0  10140.0  Aberdeen, WA                                  ...   
3  101.0  10180.0  Abilene, TX                                   ...   
4  999.0  10220.0  Ada, OK                                       ...   

   Metro /Micro Code   Total  1 Unit  2 Units  3 and 4 Units  5 Units or More  \
0                NaN     NaN     NaN      NaN            NaN              NaN   
1                5.0   169.0    65.0     24.0            0.0             80.0   
2                5.0   224.0   201.0      6.0           17.0              0.0   
3                2.0  1160.0   834.0    326.0            0.0              0.0   
4                5.0    27.0    15.0     12.0            0.0              0.0   

   Num of Structures With 5 Units or More  
0                                   

In [2]:
charleston_permits = df_permits[
    df_permits["CBSA"] == 16700
].copy()

print(charleston_permits)

       CSA     CBSA                                               Name  \
158  999.0  16700.0  Charleston-North Charleston, SC               ...   

     Metro /Micro Code   Total  1 Unit  2 Units  3 and 4 Units  \
158                4.0  7287.0  5799.0     38.0          113.0   

     5 Units or More  Num of Structures With 5 Units or More  
158           1337.0                                   101.0  


In [4]:
print("\nMetro / Micro Code Counts")
print(
    df_permits["Metro /Micro Code"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample Areas by Metro / Micro Code")
print(
    df_permits[
        ["CBSA", "Name", "Metro /Micro Code", "Total"]
    ]
    .dropna(subset=["CBSA"])
    .groupby("Metro /Micro Code")
    .head(5)
    .to_string(index=False)
)


Metro / Micro Code Counts
Metro /Micro Code
2.0    281
4.0    112
5.0    538
NaN      1
Name: count, dtype: int64

Sample Areas by Metro / Micro Code
   CBSA                                                         Name  Metro /Micro Code  Total
10100.0 Aberdeen, SD                                                                5.0  169.0
10140.0 Aberdeen, WA                                                                5.0  224.0
10180.0 Abilene, TX                                                                 2.0 1160.0
10220.0 Ada, OK                                                                     5.0   27.0
10300.0 Adrian, MI                                                                  5.0  184.0
10380.0 Aguadilla, PR                                                               2.0  716.0
10420.0 Akron, OH                                                                   2.0  751.0
10460.0 Alamogordo, NM                                                              5.0  

In [5]:
permits_metro = df_permits[
    df_permits["Metro /Micro Code"].isin([2, 4])
].copy()

print("\nMetropolitan Areas Only")
print(permits_metro.head())

print("\nNumber of Metropolitan Areas:")
print(len(permits_metro))

print("\nUnique CBSAs:")
print(permits_metro["CBSA"].nunique())

print("\nMissing Total Permit Values:")
print(permits_metro["Total"].isna().sum())


Metropolitan Areas Only
      CSA     CBSA                                               Name  \
3   101.0  10180.0  Abilene, TX                                   ...   
6   364.0  10380.0  Aguadilla, PR                                 ...   
7   184.0  10420.0  Akron, OH                                     ...   
10  999.0  10500.0  Albany, GA                                    ...   
11  440.0  10540.0  Albany, OR                                    ...   

    Metro /Micro Code   Total  1 Unit  2 Units  3 and 4 Units  \
3                 2.0  1160.0   834.0    326.0            0.0   
6                 2.0   716.0   519.0     16.0           13.0   
7                 2.0   751.0   638.0      0.0           35.0   
10                4.0   192.0   154.0      2.0            0.0   
11                2.0   189.0   140.0     20.0            3.0   

    5 Units or More  Num of Structures With 5 Units or More  
3               0.0                                     0.0  
6             168.0  

In [6]:
print("\n2025 Building Permit Statistics")
print(permits_metro["Total"].describe())

print("\nAdditional Building Permit Statistics")
print("Median:", permits_metro["Total"].median())
print("Variance:", permits_metro["Total"].var())
print("Skewness:", permits_metro["Total"].skew())

print("\nHighest Total Permit Values")
print(
    permits_metro[
        ["Name", "CBSA", "Total"]
    ]
    .sort_values("Total", ascending=False)
    .head(10)
    .to_string(index=False)
)


2025 Building Permit Statistics
count      393.000000
mean      3307.139949
std       7060.762203
min          5.000000
25%        406.000000
50%       1017.000000
75%       2993.000000
max      66179.000000
Name: Total, dtype: float64

Additional Building Permit Statistics
Median: 1017.0
Variance: 49854362.88087448
Skewness: 5.264501044442757

Highest Total Permit Values
                                                        Name    CBSA   Total
Dallas-Fort Worth-Arlington, TX                              19100.0 66179.0
Houston-Pasadena-The Woodlands, TX                           26420.0 65075.0
New York-Newark-Jersey City, NY-NJ                           35620.0 44444.0
Phoenix-Mesa-Chandler, AZ                                    38060.0 39699.0
Atlanta-Sandy Springs-Roswell, GA                            12060.0 33670.0
Orlando-Kissimmee-Sanford, FL                                36740.0 27845.0
Austin-Round Rock-San Marcos, TX                             12420.0 27322.0
Los Ange